In [49]:
import sys
import glob
import os
from os.path import join
sys.path.append('../classifier')
import pandas as pd
import numpy as np
from torchvision import transforms
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from chexpert_binary_classifier import CheXpertClassifier
from tqdm import tqdm

STYLE = "pleural_effusion"
OTHER_STYLE = "support_devices"
IMAGE_FOLDER = f'/usr/local/data/zahrat/workshop/dent_output/chexpert-swap-normal-custom2/{STYLE}'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load the model

if STYLE == 'support_devices':
    checkpoint_1 = torch.load('../saved_models/chexpert-efficientnet/binary_classifier_support-devices.pt/best_model_512_2025-02-21_18-06-55.pth')
    checkpoint_2 = torch.load('../saved_models/chexpert-efficientnet/binary_classifier_pleural-effusion.pt/best_model_512_2025-02-21_18-05-59.pth')
elif STYLE == 'pleural_effusion':
    checkpoint_2 = torch.load('../saved_models/chexpert-efficientnet/binary_classifier_support-devices.pt/best_model_512_2025-02-21_18-06-55.pth')
    checkpoint_1 = torch.load('../saved_models/chexpert-efficientnet/binary_classifier_pleural-effusion.pt/best_model_512_2025-02-21_18-05-59.pth')
else:
    raise ValueError(f"Invalid style: {STYLE}")

model_1 = CheXpertClassifier()
model_1.load_state_dict(checkpoint_1['model_state_dict'])
model_1.eval()
model_1.to(DEVICE)

model_2 = CheXpertClassifier()
model_2.load_state_dict(checkpoint_2['model_state_dict'])
model_2.eval()
model_2.to(DEVICE)


/tmp/ipykernel_1920897/175052413.py:26: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint_2 = torch.load('../saved_models/chexpert-efficientnet/binary_classifier_supp

CheXpertClassifier(
  (model): EfficientNet(
    (features): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): SiLU(inplace=True)
      )
      (1): Sequential(
        (0): MBConv(
          (block): Sequential(
            (0): Conv2dNormActivation(
              (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
              (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
              (2): SiLU(inplace=True)
            )
            (1): SqueezeExcitation(
              (avgpool): AdaptiveAvgPool2d(output_size=1)
              (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
              (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
              (activation): SiLU(inplace=True)
              (sc

In [50]:
class CustomDataset(Dataset):
    def __init__(self, img_paths, transform=None):
        self.img_paths = img_paths 
        self.transform = transform
    
    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        img = Image.open(img_path).convert('RGB')
        if self.transform is not None:
            img = self.transform(img)
        return img, img_path

    def __len__(self):
        return len(self.img_paths)

transform = transforms.Compose([
        transforms.Resize((512, 512)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])


seeds = os.listdir(IMAGE_FOLDER)

lambda_t_stars = [f'lambda_t_star{i}' for i in range(5, 46, 5)]

style_image_paths = [join(IMAGE_FOLDER, seed, lambda_t_star, 'style.png') for seed in seeds for lambda_t_star in lambda_t_stars]
style_image_paths = [path for path in style_image_paths if os.path.exists(path)]

# Load the dataset
dataset = CustomDataset(style_image_paths, transform=transform)
test_loader = DataLoader(dataset, batch_size=64, num_workers=8, 
                           pin_memory=True)

In [51]:
patients_info = {}

for i, (img, img_path) in tqdm(enumerate(test_loader), total=len(test_loader)):
    img = img.to(DEVICE)
    with torch.no_grad():
        output = model_1(img)
    output = torch.sigmoid(output)
    output = output.cpu().numpy()
    for j, path in enumerate(img_path):
        seed = path.split('/')[-3]
        lambda_t_star = path.split('/')[-2]
        
        
        if seed not in patients_info:
            patients_info[seed] = {'seed': seed[4:], 'path': path}
            
            neutral_path = join(IMAGE_FOLDER, seed, 'lambda_t_star5', 'neutral.png')
            img = Image.open(neutral_path).convert('RGB')
            img = transform(img).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                neutral_output = model_1(img)
            neutral_output = torch.sigmoid(neutral_output)
            neutral_output = neutral_output.cpu().numpy()
            
            patients_info[seed][f'neutral_{STYLE}'] = neutral_output[0][0]
            
        patients_info[seed][f'{lambda_t_star[13:]}_{STYLE}'] = output[j][0]

100%|██████████| 15/15 [00:06<00:00,  2.49it/s]


In [52]:
for i, (img, img_path) in tqdm(enumerate(test_loader), total=len(test_loader)):
    img = img.to(DEVICE)
    with torch.no_grad():
        output = model_2(img)
    output = torch.sigmoid(output)
    output = output.cpu().numpy()
    for j, path in enumerate(img_path):
        seed = path.split('/')[-3]
        lambda_t_star = path.split('/')[-2]
        
        
        if seed not in patients_info:
            patients_info[seed] = {'seed': seed[4:], 'path': path}
        
        if f'neutral_{OTHER_STYLE}' not in patients_info[seed]:
            neutral_path = join(IMAGE_FOLDER, seed, 'lambda_t_star5', 'neutral.png')
            img = Image.open(neutral_path).convert('RGB')
            img = transform(img).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                neutral_output = model_2(img)
            neutral_output = torch.sigmoid(neutral_output)
            neutral_output = neutral_output.cpu().numpy()
            
            patients_info[seed][f'neutral_{OTHER_STYLE}'] = neutral_output[0][0]
            
        patients_info[seed][f'{lambda_t_star[13:]}_{OTHER_STYLE}'] = output[j][0]

100%|██████████| 15/15 [00:05<00:00,  2.80it/s]


In [53]:
df = pd.DataFrame.from_records(list(patients_info.values()), index='seed')
df.to_csv(f'../metrics/{STYLE}_predictions_custom2.csv', index=True)

In [54]:
df


,path,neutral_pleural_effusion,5_pleural_effusion,10_pleural_effusion,15_pleural_effusion,20_pleural_effusion,25_pleural_effusion,30_pleural_effusion,35_pleural_effusion,40_pleural_effusion,...,neutral_support_devices,5_support_devices,10_support_devices,15_support_devices,20_support_devices,25_support_devices,30_support_devices,35_support_devices,40_support_devices,45_support_devices
seed,,,,,,,,,,,,,,,,,,,,,
13,/usr/local/data/zahrat/workshop/dent_output/ch...,0.045792,0.085715,0.104056,0.068742,0.066992,0.064031,0.065801,0.058658,0.054829,...,0.013678,0.020167,0.017109,0.016140,0.016807,0.015904,0.014500,0.014967,0.015196,0.014778
91,/usr/local/data/zahrat/workshop/dent_output/ch...,0.066136,0.249838,0.236392,0.186563,0.166875,0.130075,0.100138,0.088637,0.082050,...,0.043386,0.606357,0.691399,0.660433,0.101728,0.049605,0.048975,0.045719,0.045088,0.044698
54,/usr/local/data/zahrat/workshop/dent_output/ch...,0.107818,0.966879,0.942129,0.849476,0.816298,0.471370,0.274706,0.229757,0.187621,...,0.280499,0.670988,0.844131,0.844633,0.839158,0.838772,0.663980,0.428517,0.385295,0.314081
30,/usr/local/data/zahrat/workshop/dent_output/ch...,0.084068,0.927884,0.702081,0.681094,0.592263,0.585357,0.403075,0.267601,0.184994,...,0.473631,0.718229,0.912073,0.927530,0.888618,0.901553,0.920887,0.901978,0.800605,0.586750
78,/usr/local/data/zahrat/workshop/dent_output/ch...,0.053133,0.262484,0.180178,0.123163,0.112087,0.111718,0.104048,0.084240,0.073446,...,0.018080,0.029738,0.025897,0.027424,0.023001,0.020150,0.019218,0.019998,0.019608,0.018523
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71,/usr/local/data/zahrat/workshop/dent_output/ch...,0.047011,0.729647,0.521751,0.352268,0.239516,0.144179,0.098642,0.074596,0.061258,...,0.050904,0.828127,0.767496,0.793610,0.720309,0.096084,0.066566,0.060310,0.058063,0.054079
66,/usr/local/data/zahrat/workshop/dent_output/ch...,0.102538,0.195074,0.180348,0.135839,0.176884,0.185472,0.160095,0.162364,0.140602,...,0.026573,0.048513,0.034300,0.028640,0.022845,0.023729,0.023247,0.024868,0.026537,0.026886
32,/usr/local/data/zahrat/workshop/dent_output/ch...,0.084806,0.283480,0.347808,0.263045,0.192010,0.141550,0.122750,0.107158,0.093832,...,0.048754,0.036875,0.046900,0.047771,0.046348,0.042504,0.044279,0.050420,0.051292,0.047878


# Analysis 

### Analysing for Pleural Effusion

In [83]:
df = pd.read_csv('../metrics/pleural_effusion_predictions_np.csv', index_col='seed')

pe_traj = ((df['5_pleural_effusion'] - df['neutral_pleural_effusion']) +
            (df['10_pleural_effusion'] - df['neutral_pleural_effusion']) +
            (df['15_pleural_effusion'] - df['neutral_pleural_effusion']) ) / 3
sd_traj = (df['5_support_devices'] - df['neutral_support_devices'] + 
            df['10_support_devices'] - df['neutral_support_devices'] +
            df['15_support_devices'] - df['neutral_support_devices']) / 3



In [102]:
(pe_traj - sd_traj > 0).sum() / df.shape[0]

0.7812

### Analysing for Support Devices

In [103]:
df = pd.read_csv('../metrics/support_devices_predictions.csv', index_col='seed')

pe_traj = ((df['5_pleural_effusion'] - df['neutral_pleural_effusion']) +
            (df['10_pleural_effusion'] - df['neutral_pleural_effusion']) +
            (df['15_pleural_effusion'] - df['neutral_pleural_effusion']) ) / 3
sd_traj = (df['5_support_devices'] - df['neutral_support_devices'] + 
            df['10_support_devices'] - df['neutral_support_devices'] +
            df['15_support_devices'] - df['neutral_support_devices']) / 3

In [107]:
(sd_traj - pe_traj > 0).sum() / df.shape[0]

0.8880393227744402